# Adaptive KV-Cache Compression & Speculative Decoding (`Qwen2.5-0.5B` -> `Qwen2.5-7B`)
**Delhi Technological University — Major Project (Group A5: Rudransh & Saksham Thakur)**

This notebook demonstrates the end-to-end two-pronged inference optimization framework:
1. **Adaptive KV-Cache Compression**: Attention Sink preservation + Heavy-Hitter retention + Dynamic `INT4`/`INT8` mid-tier quantization + Token Pruning.
2. **Speculative Decoding Pipeline**: `Qwen/Qwen2.5-0.5B` (Draft) + `Qwen/Qwen2.5-7B` (Target) with synchronized KV-cache rollback (`crop`) and confidence-aware dynamic lookahead (`adaptive_gamma`).

In [ ]:
# Install required dependencies inside the active environment
%pip install -q -r ../requirements.txt

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from config import AdaptiveCacheConfig, BenchmarkConfig, ModelConfig, SpeculativeConfig
from src.models.loader import load_model_pair, create_tiny_qwen_pair_for_testing
from src.benchmark.runner import run_ablation_benchmark
from IPython.display import Image, display

## 1. Load Draft (`Qwen/Qwen2.5-0.5B`) & Target (`Qwen/Qwen2.5-7B`) Models
Set `USE_MOCK = False` to load the full Hugging Face weights (FP16 target by default; set `target_quant='bnb4'` only if `Qwen2.5-7B` + `Qwen2.5-0.5B` must fit on GPUs with 8–16 GB VRAM).

In [ ]:
USE_MOCK = False  # Set True for instant architecture verification without downloading 15GB weights

if USE_MOCK:
    models = create_tiny_qwen_pair_for_testing()
else:
    model_cfg = ModelConfig(
        draft_model_name='Qwen/Qwen2.5-0.5B',
        target_model_name='Qwen/Qwen2.5-7B',
        dtype='float16',
        target_quant='fp16',  # FP16 target by default (use 'bnb4' only on <=12GB GPUs)
    )
    models = load_model_pair(model_cfg)

## 2. Run 4-Way Ablation Benchmark (Throughput, Latency, Memory, Perplexity, Energy/Token)

In [ ]:
cache_cfg = AdaptiveCacheConfig(
    enabled=True,
    max_cache_size=256 if not USE_MOCK else 32,
    num_sink_tokens=4,
    recent_window_size=64 if not USE_MOCK else 10,
    heavy_hitter_ratio=1.0,  # warm-tier INT4/INT8 OFF (pure pruning)
    quant_bits=4,
)

spec_cfg = SpeculativeConfig(
    enabled=True,
    gamma=5,
    adaptive_gamma=True,
    temperature=0.0,
)

bench_cfg = BenchmarkConfig(
    max_new_tokens=128 if not USE_MOCK else 40,
    output_dir='../results',
)

report = run_ablation_benchmark(
    models=models,
    bench_config=bench_cfg,
    cache_config=cache_cfg,
    spec_config=spec_cfg,
)

display(Image(filename=report['plot_path']))